In [1]:
import os
%pwd
os.chdir("../")
%pwd

'd:\\Data Science\\END to END Proj\\DrugToxicity'

In [9]:
## ENTITY
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path           # Input: artifacts/data_ingestion/tox21.csv
    features_path: Path       # Output: artifacts/data_transformation/X.npy
    labels_path: Path         # Output: artifacts/data_transformation/y.npy
    selector_path: Path       # Output: artifacts/data_transformation/selector.pkl
    scaler_path: Path         # Output: artifacts/data_transformation/scaler.pkl

In [10]:
from src.DrugToxicity.constant import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.DrugToxicity.utils.common import read_yaml, create_directories

In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            features_path=Path(config.features_path),
            labels_path=Path(config.labels_path),
            selector_path=Path(config.selector_path),
            scaler_path=Path(config.scaler_path)
        )

In [12]:
import os
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import VarianceThreshold
from src.DrugToxicity.utils.common import logger

## RDKit imports
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, rdMolDescriptors
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

TARGET_COLS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

In [13]:
def smiles_to_features(smiles: str) -> np.ndarray:
    """
    Convert SMILES → 4719-dimensional feature vector:
      1. Morgan ECFP4     (2048 bits)
      2. MACCS keys       ( 167 bits)
      3. RDKit path FP    (2048 bits)
      4. Topological tors ( 256 bits)
      5. RDKit descriptors( 200 floats)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(4719, dtype=np.float32)

    morgan  = np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048), dtype=np.float32)
    maccs   = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32)
    rdkitfp = np.array(Chem.RDKFingerprint(mol, fpSize=2048), dtype=np.float32)
    torsion = np.array(
        rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=256),
        dtype=np.float32
    )

    desc_vals = []
    for name, _ in Descriptors.descList[:200]:
        try:
            val = float(getattr(Descriptors, name)(mol))
            if not np.isfinite(val):
                val = 0.0
        except Exception:
            val = 0.0
        desc_vals.append(val)

    feat = np.concatenate([morgan, maccs, rdkitfp, torsion, np.array(desc_vals, dtype=np.float32)])
    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(feat, -1e4, 1e4).astype(np.float32)

In [14]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def transform(self):
        """
        Skips entirely if X.npy already exists in artifacts/data_transformation.
        Otherwise:
          1. Builds 4719-dim molecular fingerprint + descriptor features
          2. Removes near-zero variance features (VarianceThreshold 0.01)
          3. Scales with RobustScaler (sparse-safe, with_centering=False)
          4. Saves X.npy, y.npy, selector.pkl, scaler.pkl
        """
        if self.config.features_path.exists():
            logger.info(
                f"Transformed features already exist at {self.config.features_path} "
                "— skipping transformation."
            )
            return False

        logger.info("Starting data transformation ...")
        os.makedirs(self.config.root_dir, exist_ok=True)

        ## Load data
        df = pd.read_csv(self.config.data_path)
        df = df.dropna(subset=["smiles"]).reset_index(drop=True)
        logger.info(f"Loaded {len(df)} compounds after dropping null SMILES.")

        ## Build raw feature matrix
        logger.info("Engineering features (Morgan + MACCS + RDKit FP + Torsion + Descriptors) ...")
        X_raw = np.vstack(df["smiles"].apply(smiles_to_features).values)
        logger.info(f"Raw feature matrix: {X_raw.shape}")

        ## Remove near-zero variance features
        selector = VarianceThreshold(threshold=0.01)
        X_sel = selector.fit_transform(X_raw)
        logger.info(f"After VarianceThreshold(0.01): {X_sel.shape}")

        ## Scale — RobustScaler, sparse-safe (with_centering=False)
        scaler = RobustScaler(with_centering=False)
        X_scaled = scaler.fit_transform(X_sel)
        X_scaled = np.clip(X_scaled, -10, 10).astype(np.float32)
        logger.info(f"Scaled & clipped. Final shape: {X_scaled.shape}")

        ## Build target matrix — NaN → -1 sentinel (masked loss pattern)
        y = np.where(
            np.isnan(df[TARGET_COLS].values.astype(float)),
            -1,
            df[TARGET_COLS].values.astype(float)
        )
        logger.info(f"Target matrix shape: {y.shape}")

        ## Save outputs
        np.save(self.config.features_path, X_scaled)
        np.save(self.config.labels_path,   y)
        joblib.dump(selector, self.config.selector_path)
        joblib.dump(scaler,   self.config.scaler_path)

        logger.info(
            f"Saved: X.npy{X_scaled.shape}, y.npy{y.shape}, "
            f"selector.pkl, scaler.pkl → {self.config.root_dir}"
        )
        return True

In [15]:
try:
    config = ConfigurationManager()
    transform_config = config.get_data_transformation_config()
    transformer = DataTransformation(config=transform_config)
    transformer.transform()
except Exception as e:
    print(f"Error during transformation: {e}")
    raise e

[2026-03-27 00:14:21,607: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-27 00:14:21,623: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-27 00:14:21,629: INFO: common: created directory at: artifacts]
[2026-03-27 00:14:21,632: INFO: common: created directory at: artifacts/data_transformation]
[2026-03-27 00:14:21,638: INFO: 4286858745: Starting data transformation ...]


[2026-03-27 00:14:21,777: INFO: 4286858745: Loaded 7831 compounds after dropping null SMILES.]
[2026-03-27 00:14:21,778: INFO: 4286858745: Engineering features (Morgan + MACCS + RDKit FP + Torsion + Descriptors) ...]


C:\Users\dell\AppData\Local\Temp\ipykernel_11148\2053075537.py:32: RuntimeWarning: overflow encountered in cast
  feat = np.concatenate([morgan, maccs, rdkitfp, torsion, np.array(desc_vals, dtype=np.float32)])


[2026-03-27 00:17:50,389: INFO: 4286858745: Raw feature matrix: (7831, 4719)]
[2026-03-27 00:17:52,223: INFO: 4286858745: After VarianceThreshold(0.01): (7831, 3133)]
[2026-03-27 00:17:53,343: INFO: 4286858745: Scaled & clipped. Final shape: (7831, 3133)]
[2026-03-27 00:17:53,363: INFO: 4286858745: Target matrix shape: (7831, 12)]
[2026-03-27 00:17:53,683: INFO: 4286858745: Saved: X.npy(7831, 3133), y.npy(7831, 12), selector.pkl, scaler.pkl → artifacts\data_transformation]


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 112: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Data Science\END to END Proj\DrugToxicity\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\Data Science\END to END Proj\DrugToxicity\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\Da